In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import LineString, MultiLineString
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Bidirectional, Dense, TimeDistributed, Concatenate, RepeatVector, Dropout, LayerNormalization
from tensorflow.keras.callbacks import ReduceLROnPlateau, ModelCheckpoint, EarlyStopping
from  tensorflow.keras.optimizers import Adam, AdamW
from tensorflow.keras.regularizers import l2
from keras.utils import custom_object_scope, timeseries_dataset_from_array
import tensorflow as tf
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from datetime import datetime

# Helper Functions

In [ ]:
#Function to predict the new sequences for line a and line b
def predict_reconstructed_sequences(network, sequence_a, sequence_b):
    #wegen Siamese werden zwei lines gleichzeitig predicted
    combined_prediction = network.predict([sequence_a, sequence_b])
    half = combined_prediction.shape[1] // 2

    #slices the combined_prediction array to extract the first half columns
    reconstructed_sequence_a = combined_prediction[:, :half]
    #slices the combined_prediction array to extract the last half columns
    reconstructed_sequence_b = combined_prediction[:, half:]

    return reconstructed_sequence_a, reconstructed_sequence_b

In [ ]:
def plot_lines(lines_a_nopad, lines_b_nopad, n, pred_line_a, pred_line_b, title, path):
    # Extract the 4 LineStrings
    line_a = LineString(lines_a_nopad[n, :, :])
    line_b = LineString(lines_b_nopad[n, :, :])
    line_pred_a = LineString(pred_line_a[n, :, :])
    line_pred_b = LineString(pred_line_b[n, :, :])

    # Start the plot
    fig, ax = plt.subplots(figsize=(6, 6))

    # Plot each line with a label
    ax.plot(*line_a.xy, label='Line A (GT)', color='blue')
    ax.plot(*line_b.xy, label='Line B (GT)', color='green')
    ax.plot(*line_pred_a.xy, label='Predicted A', color='red', linestyle='--')
    ax.plot(*line_pred_b.xy, label='Predicted B', color='orange', linestyle='--')

    # Add styling
    ax.legend()
    ax.set_title(title)
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")
    ax.grid(True)

    # Save and show
    plt.savefig(path, dpi=300, bbox_inches='tight')
    plt.show()

# Model Functions

In [ ]:
def perp_distance(args):

    point_a1, point_a2, point_b = args
    # Direction vector of the segment
    v = point_a2 - point_a1                     # (..., 2)
    # Vector from A1 to the query point
    w = point_b  - point_a1                     # (..., 2)

    # Squared length of the segment ‖v‖²  (add ε for numerical safety)
    vv = tf.reduce_sum(tf.square(v), axis=-1, keepdims=True) + tf.keras.backend.epsilon()

    # Projection scalar t = (w·v) / (‖v‖²)  — clip to [0,1] to stay on the *segment*
    t = tf.reduce_sum(w * v, axis=-1, keepdims=True) / vv
    t_clipped = tf.clip_by_value(t, 0.0, 1.0)  # (..., 1)

    # Nearest point on the segment to point_b
    nearest = point_a1 + t_clipped * v         # (..., 2)

    # Euclidean distance ‖point_b − nearest‖
    return tf.norm(point_b - nearest, axis=-1) # (...,)


In [ ]:
# # #Abstand zwischen Punkt (Predicted) und Linie (Original)
# def perp_distance(args):
#     point_a1, point_a2, point_b = args
#     lvs = point_a1 #Start des Richtungsvektors Line Segment A
#     lve = point_a2 #Ende des Richtungsvektors Line Segment A
#     dvc = lve - lvs #Richtungsvektor
#     pvc = point_b #Punkt, zu dem Abstand berechnet werden soll

#     #If the segment is a single point (start = end), simply compute the Euclidean distance from pvc to lvs
#     def case_equal():
#         return tf.norm(pvc - lvs)

#     def case_unequal():
#         tvc = lvs - pvc #vector from point to segment start.
#         tm0 = tvc * dvc #dot products used to compute projection scalar lbd
#         tm1 = dvc * dvc
#         lbd = -tf.reduce_sum(tm0) / (tf.reduce_sum(tm1) + tf.keras.backend.epsilon()) #tells how far along the line the projection falls, small epsilon added to avoid division by zero
#         nvc = lvs + lbd * dvc #nearest point on the line segment (can be outside the actual segment).
#         dist_inside = tf.norm(nvc - pvc)
#         dist_start = tf.norm(nvc - lvs)
#         dist_end = tf.norm(nvc - lve)
#         condition_inside = tf.logical_and(tf.greater(lbd, 0), tf.less(lbd, 1)) #condition_inside: projection inside the segment.
#         condition_start = tf.less_equal(lbd, 0) #condition_start: projection before start (otherwise after end).
#         return tf.where(condition_inside, dist_inside, tf.where(condition_start, dist_start, dist_end))

#     condition = tf.reduce_all(tf.equal(lvs, lve))

#     #If inside, use the perpendicular distance, Otherwise, use distance to nearest endpoint.
#     return tf.cond(condition, case_equal, case_unequal)

In [ ]:
#Berechnet für jeden predicted Point B die Perp_Distance zu Point original B
def displacement_loss(y_pred_b, y_true_a):

    min_distances = []
    for point_b in tf.unstack(y_pred_b, axis=1):
        dists = tf.map_fn(perp_distance, (y_true_a[:, :-1, :], y_true_a[:, 1:, :], point_b), dtype=tf.float32)
        min_dist = tf.reduce_min(dists)
        min_distances.append(min_dist)

    return tf.reduce_mean(min_distances)

In [ ]:
# # #Berechnet den Loss zwischen True A und Pred A, True B und Pred B sowie den Displacement Loss
# def combined_loss(y_true_a, y_pred_a, y_pred_b, y_true_b, alpha):

#     #Compute mean-squared error (MSE) between true and predicted A points, and true and predicted B points.
#     mse_loss_a = tf.reduce_mean(tf.keras.losses.MSE(y_true_a, y_pred_a))
#     mse_loss_b = tf.reduce_mean(tf.keras.losses.MSE(y_true_b, y_pred_b))

#     #Compute displacement loss between predicted B points and original A line.
#     disp_loss = displacement_loss(y_pred_b, y_true_a) #THIS CAN BE MODIFIED displacement of b, (y_pred_b, y_pred_a) --> displacement of both lines

#     tf.print(' MSE loss:', mse_loss_a+mse_loss_b, ' Neg Displacement:',- (alpha * disp_loss))

#     return mse_loss_a + mse_loss_b - (alpha * disp_loss)

In [ ]:
# # #you want to pass in alpha dynamically when you build the model, but Keras loss functions can only take (y_true, y_pred) when called during training.
# def siamese_loss_wrapper(alpha):
#     def siamese_loss(y_true, y_pred):
#         last_dim = tf.shape(y_true)[1]  # shape is (batch_size, last_dim)
#         half = last_dim // 2

#         # Split y_true and y_pred dynamically
#         y_true_a = y_true[:, :half]
#         y_true_b = y_true[:, half:]
#         y_pred_a = y_pred[:, :half]
#         y_pred_b = y_pred[:, half:]

#         return combined_loss(y_true_a, y_pred_a, y_pred_b, y_true_b, alpha)
#     return siamese_loss

In [ ]:
def combined_loss(y_true_a, y_pred_a, y_pred_b, y_true_b, alpha):
    y_true_a = tf.cast(y_true_a, tf.float32)
    y_true_b = tf.cast(y_true_b, tf.float32)
    y_pred_a = tf.cast(y_pred_a, tf.float32)
    y_pred_b = tf.cast(y_pred_b, tf.float32)

    mse_loss_a = tf.reduce_mean(tf.square(y_true_a - y_pred_a))
    mse_loss_b = tf.reduce_mean(tf.square(y_true_b - y_pred_b))

    disp_loss = displacement_loss(y_pred_b, y_true_a)

    # Optional debug
    tf.print('MSE A:', mse_loss_a, 'MSE B:', mse_loss_b, 'Disp:', disp_loss)

    return mse_loss_a + mse_loss_b - (alpha * disp_loss)

def siamese_loss_wrapper(alpha):
    def siamese_loss(y_true, y_pred):
        num_points = tf.shape(y_pred)[1] // 2

        y_true_a = y_true[:, :num_points, :]
        y_true_b = y_true[:, num_points:, :]
        y_pred_a = y_pred[:, :num_points, :]
        y_pred_b = y_pred[:, num_points:, :]

        return combined_loss(y_true_a, y_pred_a, y_pred_b, y_true_b, alpha)
    return siamese_loss

In [ ]:
# apply LayerNormalization()(x) for each decoder / encoder
# def create_autoencoder(input_shape, prefix):
#     # Define Encoder
#     inputs = Input(shape=input_shape)
#     encoder_bi_lstm1 = Bidirectional(LSTM(128, return_sequences=True))(inputs)
#     encoder_bi_lstm2 = Bidirectional(LSTM(64, return_sequences=True))(encoder_bi_lstm1)
#     encoder_bi_lstm3 = Bidirectional(LSTM(32, return_sequences=True, return_state=True))
#     encoder_outputs, forward_h, forward_c, backward_h, backward_c = encoder_bi_lstm3(encoder_bi_lstm2)
#     state_h = Concatenate()([forward_h, backward_h])
#     state_c = Concatenate()([forward_c, backward_c])
#     encoder_states = [state_h, state_c]

#     # Define Decoder
#     forward_decoder_lstm = LSTM(32, return_sequences=True)
#     backward_decoder_lstm = LSTM(32, return_sequences=True, go_backwards=True)
#     forward_decoder_outputs = forward_decoder_lstm(encoder_outputs, initial_state=[forward_h, forward_c])
#     backward_decoder_outputs = backward_decoder_lstm(encoder_outputs, initial_state=[backward_h, backward_c])

#     # Combine Forward and Backward LSTM
#     decoder_outputs = Concatenate()([forward_decoder_outputs, backward_decoder_outputs])
#     decoder_bi_lstm2 = Bidirectional(LSTM(64, return_sequences=True))(decoder_outputs)
#     decoder_bi_lstm3 = Bidirectional(LSTM(128, return_sequences=True))(decoder_bi_lstm2)

#     decoder_dense = TimeDistributed(Dense(2, activation='linear')) #Conv1D(filters=2, kernel_size=1)
#     decoder_outputs = decoder_dense(decoder_bi_lstm3)
#     autoencoder = Model(inputs, decoder_outputs)

#     return autoencoder

In [ ]:
def create_autoencoder(input_shape, prefix):

    inputs = Input(shape=input_shape)

    # Encoder
    encoder_bi_lstm1 = Bidirectional(LSTM(128, return_sequences=True))(inputs)
    encoder_bi_lstm1 = LayerNormalization()(encoder_bi_lstm1)
    encoder_bi_lstm1 = Dropout(0.3)(encoder_bi_lstm1)

    encoder_bi_lstm2 = Bidirectional(LSTM(64, return_sequences=True))(encoder_bi_lstm1)
    encoder_bi_lstm2 = LayerNormalization()(encoder_bi_lstm2)
    encoder_bi_lstm2 = Dropout(0.3)(encoder_bi_lstm2)

    encoder_bi_lstm3 = Bidirectional(LSTM(32, return_sequences=True, return_state=True))
    encoder_outputs, forward_h, forward_c, backward_h, backward_c = encoder_bi_lstm3(encoder_bi_lstm2)
    state_h = Concatenate()([forward_h, backward_h])
    state_c = Concatenate()([forward_c, backward_c])
    encoder_states = [state_h, state_c]

    encoder_outputs = LayerNormalization()(encoder_outputs)
    encoder_outputs = Dropout(0.3)(encoder_outputs)

    # Decoder
    forward_decoder_lstm = LSTM(32, return_sequences=True)
    backward_decoder_lstm = LSTM(32, return_sequences=True, go_backwards=True)

    forward_decoder_outputs = forward_decoder_lstm(encoder_outputs, initial_state=[forward_h, forward_c])
    backward_decoder_outputs = backward_decoder_lstm(encoder_outputs, initial_state=[backward_h, backward_c])

    decoder_outputs = Concatenate()([forward_decoder_outputs, backward_decoder_outputs])
    decoder_outputs = LayerNormalization()(decoder_outputs)
    decoder_outputs = Dropout(0.3)(decoder_outputs)

    decoder_bi_lstm2 = Bidirectional(LSTM(64, return_sequences=True))(decoder_outputs)
    decoder_bi_lstm2 = LayerNormalization()(decoder_bi_lstm2)
    decoder_bi_lstm2 = Dropout(0.3)(decoder_bi_lstm2)

    decoder_bi_lstm3 = Bidirectional(LSTM(128, return_sequences=True))(decoder_bi_lstm2)
    decoder_bi_lstm3 = LayerNormalization()(decoder_bi_lstm3)
    decoder_bi_lstm3 = Dropout(0.3)(decoder_bi_lstm3)

    decoder_dense = TimeDistributed(Dense(2, activation='linear'))
    decoder_outputs = decoder_dense(decoder_bi_lstm3)

    autoencoder = Model(inputs, decoder_outputs)

    return autoencoder

In [ ]:
def make_siamese_dataset(a_noisy, b_noisy, a_clean, b_clean, shuffle=True, batch_size=32):
    inputs = (a_noisy, b_noisy)
    # combine clean targets into one tensor
    targets = np.stack([a_clean, b_clean], axis=1)  # shape: (N, 2, 128, 2)

    dataset = tf.data.Dataset.from_tensor_slices((inputs, targets))

    if shuffle:
        dataset = dataset.shuffle(buffer_size=1024)

    return dataset.batch(batch_size)

# **MAIN EXECUTION**

In [ ]:
PATH_TRAINING_A = '../data/final_dataset/sequence/real-world-river.npy'
PATH_TRAINING_B = '../data/final_dataset/sequence/real-world-river-displaced_close.npy'

PATH_TRAINING_A_CLEAN = '../data/final_dataset/sequence/real-world-river.npy'
PATH_TRAINING_B_CLEAN = '../data/final_dataset/sequence/real-world-river-displaced_far.npy'

# Load full arrays
lines_a_noisy = np.load(PATH_TRAINING_A)[:100]
lines_b_noisy = np.load(PATH_TRAINING_B)[:100]
lines_a_clean = np.load(PATH_TRAINING_A_CLEAN)[:100]
lines_b_clean = np.load(PATH_TRAINING_B_CLEAN)[:100]

# Total number of samples
n_total = lines_a_noisy.shape[0]

# Compute split indices
n_train = int(0.7 * n_total)
n_val = int(0.15 * n_total)
n_test = n_total - n_train - n_val

# --- TRAIN ---
train_slice = slice(0, n_train)
lines_a_noisy_train = lines_a_noisy[train_slice]
lines_b_noisy_train = lines_b_noisy[train_slice]
lines_a_clean_train = lines_a_clean[train_slice]
lines_b_clean_train = lines_b_clean[train_slice]

# --- VALIDATION ---
val_slice = slice(n_train, n_train + n_val)
lines_a_noisy_val = lines_a_noisy[val_slice]
lines_b_noisy_val = lines_b_noisy[val_slice]
lines_a_clean_val = lines_a_clean[val_slice]
lines_b_clean_val = lines_b_clean[val_slice]

# --- TEST ---
test_slice = slice(n_train + n_val, n_total)
lines_a_noisy_test = lines_a_noisy[test_slice]
lines_b_noisy_test = lines_b_noisy[test_slice]
lines_a_clean_test = lines_a_clean[test_slice]
lines_b_clean_test = lines_b_clean[test_slice]


# Create training, validation, and test datasets
train_dataset = make_siamese_dataset(lines_a_noisy, lines_b_noisy, lines_a_clean, lines_b_clean)
val_dataset   = make_siamese_dataset(lines_a_noisy_val, lines_b_noisy_val, lines_a_clean_val, lines_b_clean_val, shuffle=False)
test_dataset  = make_siamese_dataset(lines_a_clean_test, lines_b_clean_test, lines_a_noisy_test, lines_b_noisy_test, shuffle=False)

In [ ]:
print(lines_a_noisy_train.shape)
print(lines_a_noisy_val.shape)
print(lines_a_noisy_test.shape)

In [ ]:
# PATH_TRAINING_A = '/content/drive/MyDrive/ma_files/data/synthetic_old/synthetic_data_a_norm_noisy.npy'
# PATH_TRAINING_B = '/content/drive/MyDrive/ma_files/data/synthetic_old/synthetic_data_b_norm_02_noisy.npy'

# PATH_TRAINING_A_CLEAN = '/content/drive/MyDrive/ma_files/data/synthetic_old/synthetic_data_a_norm.npy'
# PATH_TRAINING_B_CLEAN = '/content/drive/MyDrive/ma_files/data/synthetic_old/synthetic_data_b_norm_10.npy'

# #TRAINING
# lines_a_noisy = np.load(PATH_TRAINING_A)[:20000, :, :]
# lines_b_noisy = np.load(PATH_TRAINING_B)[:20000, :, :]

# lines_a_clean= np.load(PATH_TRAINING_A_CLEAN)[:20000, :, :]
# lines_b_clean = np.load(PATH_TRAINING_B_CLEAN)[:20000, :, :]

# #VALIDATION
# lines_a_val_noisy = np.load(PATH_TRAINING_A)[20000:, :, :]
# lines_b_val_noisy = np.load(PATH_TRAINING_B)[20000:, :, :]

# lines_a_val_clean= np.load(PATH_TRAINING_A_CLEAN)[20000:, :, :]
# lines_b_val_clean = np.load(PATH_TRAINING_B_CLEAN)[20000:, :, :]

# #TEST
# lines_a_clean_test= np.load(PATH_TRAINING_A_CLEAN)[22500:, :, :]
# lines_b_clean_test = np.load(PATH_TRAINING_B_CLEAN)[22500:, :, :]

# lines_a_noisy_test = np.load(PATH_TRAINING_A)[22500:, :, :]
# lines_b_noisy_test = np.load(PATH_TRAINING_B)[22500:, :, :]

# lines_a_noisy.shape

In [ ]:
#Create two models, two input tensors and two reconstructed sequence tensors
input_shape = (lines_a_noisy_train.shape[1], lines_a_noisy_train.shape[2])
autoencoder_a = create_autoencoder(input_shape, 'A')
autoencoder_b = create_autoencoder(input_shape, 'B')

input_sequence_a = Input(shape=input_shape)
input_sequence_b = Input(shape=input_shape)
print(input_sequence_a)

reconstructed_sequence_a = autoencoder_a(input_sequence_a)
reconstructed_sequence_b = autoencoder_b(input_sequence_b)
print(reconstructed_sequence_a)

#Construct Model Architecture
siamese_autoencoder = Model([input_sequence_a, input_sequence_b], Concatenate(axis=1)([reconstructed_sequence_a, reconstructed_sequence_b]))

In [ ]:
#Define Parameters
loss = siamese_loss_wrapper(0.001)
epochs = 10 #50
batch_size=16 #32
timestamp = datetime.now().strftime("%H%M%S")

CHECKPOINT_CALLBACK = f'../checkpoints/siamese/weights/{batch_size}_batches_{epochs}_epochs_{timestamp}.weights.h5'

#Checkpoint callback: saves models weights, when loss is smaller than before
checkpoint_callback = ModelCheckpoint(
    filepath= CHECKPOINT_CALLBACK,
    monitor='loss',
    mode="min",
    save_best_only=True,
    save_weights_only=True,
    verbose=1,
)

early_stopping_callback = EarlyStopping(
    monitor='loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

optimizer = AdamW(
    learning_rate=1e-3,
    weight_decay=1e-4,
    clipnorm=1.0
)

siamese_autoencoder.compile(optimizer=optimizer, loss=loss, metrics=['accuracy'])
lines_combined = np.concatenate([lines_a_clean_train, lines_b_clean_train], axis=1)
val_lines_combined = np.concatenate([lines_a_clean_val, lines_b_clean_val], axis=1)

In [ ]:
history = siamese_autoencoder.fit(
    [lines_a_noisy_train, lines_b_noisy_train], lines_combined,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(
      [lines_a_noisy_val, lines_b_noisy_val], val_lines_combined
    ),
    callbacks=[checkpoint_callback])

siamese_autoencoder.load_weights(filepath=CHECKPOINT_CALLBACK)
siamese_autoencoder.save(f'../checkpoints/siamese/model/{batch_size}_batches_{epochs}_epochs_rw_newLoss_newModel_{timestamp}.keras')
np.save('../checkpoints/siamese/siamese_history.npy', history.history)

In [ ]:
siamese_autoencoder.load_weights(filepath='../checkpoints/siamese/weights/')

In [ ]:
siamese_autoencoder.summary()

In [ ]:
#plots the history/verlauf des Training loss
#history = np.load('/content/drive/MyDrive/ma_files/siamese_history.npy')

def plot_history(history, epochs, batch_size, train_data_noisy):
    plt.figure(figsize=(10, 5))
    plt.plot(history.history['loss'], label='Training Loss', color='#00305D')

    if 'val_loss' in history.history:
        plt.plot(history.history['val_loss'], label='Validation Loss', color='#65B32E')

    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.title(f'Training Loss Siamese LSTM AE {epochs} Epochs {batch_size} Batches and Training Data Shape: {train_data_noisy.shape}')
    plt.show()

plot_history(history, epochs, batch_size, lines_a_noisy)

# **Post-Training**

In [ ]:
#siamese_autoencoder = tf.keras.models.load_model(f'/content/drive/MyDrive/ma_files/models/siamese/32_batches_50_epochs_syndata_ae160306.keras', custom_objects={'siamese_loss': loss})
#siamese_autoencoder.load_weights(filepath='/content/drive/MyDrive/ma_files/checkpoints/siamese/32_batches_50_epochs_rw_newLoss_newModel_160340.weights.h5') #CHECKPOINT_CALLBACK

In [ ]:
pred_sequence_a= lines_a_noisy_test
pred_sequence_a_clean = lines_a_clean_test

pred_sequence_b = lines_b_noisy_test
pred_sequence_b_clean = lines_b_clean_test

In [ ]:
# lines = [LineString(coords) for coords in pred_line_b]

# gdf = gpd.GeoDataFrame(geometry=lines)
# palette = plt.cm.Set1(np.linspace(0, 1, len(gdf)))
# palette = [tuple(rgba) for rgba in palette]
# gdf.plot(figsize=(8, 8), linewidth=2, color=palette)

In [ ]:
pred_siamese_a, pred_siamese_b = predict_reconstructed_sequences(siamese_autoencoder, pred_sequence_a, pred_sequence_b)

In [ ]:
pred_siamese_b.shape

In [ ]:
plt.figure(figsize=(12, 12))
start_plot = 1300
end_plot = start_plot +1

# Plot training segments A
for i, segment in enumerate(lines_a_noisy_train[start_plot:end_plot,:,:]):
    plt.plot(segment[:, 0], segment[:, 1],
             color='#009FE3', linewidth=0.25,
             label='Original A' if i == 0 else "")
      # plt.plot(segment[0:1, 0], segment[0:1, 1],
      #     color='#009FE3', marker='o', markersize=1,
      #     label='Input A' if i == 0 else "")

# Plot training segments B
for i, segment in enumerate(lines_b_noisy_train[start_plot:end_plot,:,:]):
    plt.plot(segment[:, 0], segment[:, 1],
             color='#FFC000', linewidth=0.25,
             label='Original B' if i == 0 else "")
    # plt.plot(segment[0:1, 0], segment[0:1, 1],
    #       color='#FFC000', marker='o', markersize=1,
    #       label='Input B' if i == 0 else "")


# Plot predicted segments A
for i, segment in enumerate(lines_a_clean_train[start_plot:end_plot,:,:]):
     plt.plot(segment[:, 0], segment[:, 1],
            color='#00305D', linestyle='--', linewidth=1,
            label='Reconstructed A' if i == 0 else "")
#     # plt.plot(segment[0:1, 0], segment[0:1, 1],
#     #       color='#00305D', marker='o', markersize=1,
#     #       label='Prediction A' if i == 0 else "")

# Plot predicted segments B
for i, segment in enumerate(lines_b_clean_train[start_plot:end_plot,:,:]):
     plt.plot(segment[:, 0], segment[:, 1],
            color='#65B32E', linestyle='--', linewidth=1,
            label='Reconstructed B' if i == 0 else "")
    # plt.plot(segment[0:1, 0], segment[0:1, 1],
    #          color='#65B32E', marker='o', markersize=1,
    #          label='Prediction B' if i == 0 else "")

plt.title('Siamese LSTM AE Dataset')
plt.xlabel('X')
plt.ylabel('Y')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import mean_squared_error

mse = mean_squared_error(pred_sequence_b_clean.reshape(-1, 2), pred_siamese_b.reshape(-1, 2))
print(f"Mean Squared Error (MSE): {mse}")

In [ ]:
# Compute Euclidean error for all sequences and timesteps
#all_errors_a = np.linalg.norm(pred_sequence_a[:] - pred_siamese_a[:], axis=-1)
all_errors_b = np.linalg.norm(pred_sequence_b[:,0,:] - pred_siamese_b[:,0,:], axis=-1)

# Flatten all errors into one 1D array
#all_errors_flat_a = all_errors_a.flatten()  # shape: [num_sequences * sequence_length]
all_errors_flat_b = all_errors_b.flatten()


# Plot all errors as one continuous line
plt.figure(figsize=(12, 4))
#plt.plot(all_errors_flat_a, color='#003f5c', linewidth=0.1, marker='o', linestyle='-', label='Errors A')
plt.plot(all_errors_flat_b, color='#003f5c', linewidth=0.1, marker='o', linestyle='-', markersize=0.1, label='Errors B')
plt.title('Euclidean Error for sequence B')
plt.xlabel('Sequence and Index (flattened)')
plt.ylabel('Euclidean Error')
plt.grid(True)
plt.show()

In [ ]:
# Compute MSE per point (squared error averaged over features)
mse_all = np.mean((pred_sequence_b[:,0,:] - pred_siamese_b[:,0,:]) ** 2, axis=-1)  # shape: [num_sequences, sequence_length]

# Flatten for one continuous plot
mse_all_flat = mse_all.flatten()

plt.figure(figsize=(12, 4))
plt.plot(mse_all_flat, color='#65B32E', linewidth=0.1, marker='o', markersize=0.1,linestyle='-')
plt.title('MSE for all Sequences and one Point')
plt.xlabel('Index (flattened)')
plt.ylabel('Mean Squared Error')
plt.grid(True)
plt.show()